A function that is not defined for all possible values of its argument is
called a partial function. It’s not really a function in the mathematical
sense, so it doesn’t fit the standard categorical mold. It can, however, be
represented by a function that returns an embellished type optional:
```cpp
template <typename T> class optional {
    bool _isValid;
    T _value;
public:
    optional() : _isValid(false) {}
    optional(T value) : _isValid(true), _value(value) {}
    bool isValid() const { return _isValid; }
    T value() const { return _value; }
};
```

As an example, here’s the implementation of the embellished function
safe_root:
```cpp
optional<double> safe_root(double x) {
    if (x >= 0) return optional<double>{sqrt(x)};
    else return optional<double>{};
}
```

1. Construct the Kleisli category for partial functions (define composition and identity).
```cpp
std::function<optional<double>(double)> compose(
    std::function<optional<double>(double)> f,
    std::function<optional<double>(double)> g) {
    return [f, g](double x) {
        optional<double> result = f(x);
        if (result.isValid()) {
            return g(result.value());
        }
        return optional<double>{};
    };
}
```

```cpp
std::function<optional<double>(double)> identity() {
    return [](double x) {
        return optional<double>{x};
    };
}
```

2. Implement the embellished function safe_reciprocal that returns a valid reciprocal of its argument, if it’s different from zero.

```cpp
optional<double> safe_inverse(double x) {
    if (x == 0) {
        return optional<double>{};
    }
    return optional<double>{1.0 / x};
}
```

3. Compose safe_root and safe_reciprocal to implement
safe_root_reciprocal that calculates sqrt(1/x) whenever possible.

```cpp
optional<double> safe_root_inverse(double x) {
    return compose(safe_inverse, safe_root)(x);
}
```

In [2]:
!g++ optional.cpp -o optional

In [3]:
%%script bash
./optional

The square root inverse of 4 is 0.5
Cannot calculate square root inverse of -1
The square root inverse of 9 is 0.333333
Cannot calculate square root inverse of 0
The square root inverse of 16 is 0.25
